# Energy Consumption Forecasting — ML Model Training

This notebook trains a **LightGBM Gradient Boosted Decision Tree (GBDT)** model for multi-step hourly energy consumption forecasting. It matches the exact production contract expected by the FastAPI backend.

### Pipeline Overview:
1. **Data Loading & Preprocessing**: Ingests raw UCI household power consumption or preprocessed CSV and aggregates to hourly kWh.
2. **Feature Engineering**: Constructs hourly lag features, rolling statistics (mean, std), calendar features, and cyclical trigonometric encodings.
3. **Time-Series Split**: Splits the dataset into training and test sets without lookahead leakage.
4. **Model Training**: Fits a LightGBM regressor.
5. **Evaluation**: Computes MAE, RMSE, MAPE, and R² scores.
6. **Artifact Export**: Saves `lightgbm_energy_forecaster.joblib`, `model_config.json`, `feature_columns.json`, `metrics.json`, and `feature_importance.csv`.

In [ ]:
import os
import json
import csv
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

# Paths setup
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"
BACKEND_DIR = BASE_DIR / "backend"
MODELS_DIR = BACKEND_DIR / "models"
ARTIFACTS_DIR = BACKEND_DIR / "artifacts"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")

## 1. Data Ingestion

Load data from `data/energy_consumption.csv` (or `data/energy_consumption_hourly.csv`), or automatically generate a 1-year realistic hourly dataset if none exists yet.

In [ ]:
def load_or_generate_data():
    hourly_file = DATA_DIR / "energy_consumption_hourly.csv"
    raw_file = DATA_DIR / "energy_consumption.csv"
    
    if hourly_file.exists():
        print(f"Loading hourly data from {hourly_file}...")
        df = pd.read_csv(hourly_file)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        return df
    elif raw_file.exists():
        print(f"Loading raw data from {raw_file}...")
        df = pd.read_csv(raw_file)
        ts_col = "timestamp" if "timestamp" in df.columns else df.columns[0]
        df["timestamp"] = pd.to_datetime(df[ts_col])
        target_col = "energy_kwh" if "energy_kwh" in df.columns else df.columns[1]
        df["energy_kwh"] = pd.to_numeric(df[target_col], errors="coerce")
        df = df.set_index("timestamp").resample("1h").mean().interpolate(method="time").reset_index()
        return df[["timestamp", "energy_kwh"]]
    else:
        print("Generating synthetic 1-year hourly energy series...")
        timestamps = pd.date_range("2024-01-01 00:00:00", periods=8760, freq="1h")
        h = timestamps.hour.values
        daily = 0.5 * np.sin(2 * np.pi * (h - 6) / 24) + 0.8 * np.exp(-((h - 8)**2) / 4) + 1.5 * np.exp(-((h - 20)**2) / 6)
        dow = timestamps.dayofweek.values
        weekend = np.where(dow >= 5, 0.4, 0.0)
        doy = timestamps.dayofyear.values
        seasonal = 0.6 * np.cos(2 * np.pi * (doy - 20) / 365) + 0.4 * np.sin(2 * np.pi * (doy - 200) / 365)
        np.random.seed(42)
        noise = np.random.normal(0, 0.2, 8760)
        values = np.clip(1.2 + daily + weekend + seasonal + noise, 0.1, None)
        return pd.DataFrame({"timestamp": timestamps, "energy_kwh": np.round(values, 4)})

df_energy = load_or_generate_data()
df_energy.info()
df_energy.head()

## 2. Feature Engineering

Feature engineering definitions match `app.services.feature_service` exactly:
- **Lags**: 1, 2, 3, 24, 48, 168 hours
- **Rolling Means**: 3, 6, 24, 168 hours
- **Rolling Stds**: 3, 6, 24, 168 hours
- **Calendar**: hour, day_of_month, day_of_week, month, week_of_year, is_weekend
- **Cyclical**: hour_sin, hour_cos, day_of_week_sin, day_of_week_cos

In [ ]:
LAGS = (1, 2, 3, 24, 48, 168)
ROLLING_WINDOWS = (3, 6, 24, 168)
CALENDAR_FEATURES = ("hour", "day_of_month", "day_of_week", "month", "week_of_year", "is_weekend")
CYCLICAL_FEATURES = ("hour_sin", "hour_cos", "day_of_week_sin", "day_of_week_cos")

FEATURE_COLUMNS = (
    [f"lag_{k}" for k in LAGS]
    + [f"rolling_mean_{w}" for w in ROLLING_WINDOWS]
    + [f"rolling_std_{w}" for w in ROLLING_WINDOWS]
    + list(CALENDAR_FEATURES)
    + list(CYCLICAL_FEATURES)
)

def create_features(series: pd.Series, cols: list[str]) -> pd.DataFrame:
    shifted = series.shift(1)
    idx = series.index
    feat_dict = {}
    
    # Calendar
    feat_dict["hour"] = idx.hour
    feat_dict["day_of_month"] = idx.day
    feat_dict["day_of_week"] = idx.dayofweek
    feat_dict["month"] = idx.month
    feat_dict["week_of_year"] = idx.isocalendar().week.to_numpy()
    feat_dict["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    
    # Cyclical
    feat_dict["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
    feat_dict["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
    feat_dict["day_of_week_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
    feat_dict["day_of_week_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
    
    for col in cols:
        if col.startswith("lag_"):
            k = int(col.split("_")[1])
            feat_dict[col] = series.shift(k).to_numpy()
        elif col.startswith("rolling_mean_"):
            w = int(col.split("_")[2])
            feat_dict[col] = shifted.rolling(w).mean().to_numpy()
        elif col.startswith("rolling_std_"):
            w = int(col.split("_")[2])
            feat_dict[col] = shifted.rolling(w).std().to_numpy()
            
    return pd.DataFrame(feat_dict, index=idx, columns=cols)

series = df_energy.set_index("timestamp")["energy_kwh"].sort_index()
X = create_features(series, FEATURE_COLUMNS)
y = series.loc[X.index]

# Drop warmup rows where lag/rolling stats are NaN
valid_idx = ~(X.isna().any(axis=1) | y.isna())
X_clean = X[valid_idx]
y_clean = y[valid_idx]

print(f"Feature matrix shape: {X_clean.shape}")

## 3. Train / Test Split & Model Fitting

In [ ]:
split_point = int(len(X_clean) * 0.8)
X_train, X_test = X_clean.iloc[:split_point], X_clean.iloc[split_point:]
y_train, y_test = y_clean.iloc[:split_point], y_clean.iloc[split_point:]

print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = mean_absolute_percentage_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.4f} kWh")
print(f"RMSE: {rmse:.4f} kWh")
print(f"MAPE: {mape*100:.2f}%")
print(f"R2:   {r2:.4f}")

## 4. Export Production Artifacts

Saves model weights and metadata into the folders required by the backend.

In [ ]:
# 1. Save model joblib
joblib.dump(model, MODELS_DIR / "lightgbm_energy_forecaster.joblib")

# 2. Save config
config = {
    "model_name": "lightgbm_energy_forecaster",
    "model_version": "1.0.0",
    "algorithm": "LightGBM Regressor",
    "target": "energy_kwh",
    "unit": "kWh",
    "frequency": "hourly",
    "features_count": len(FEATURE_COLUMNS),
    "training_date": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
}
(ARTIFACTS_DIR / "model_config.json").write_text(json.dumps(config, indent=2))

# 3. Save feature columns
(ARTIFACTS_DIR / "feature_columns.json").write_text(json.dumps(FEATURE_COLUMNS, indent=2))

# 4. Save metrics
metrics = {
    "mae": round(float(mae), 4),
    "rmse": round(float(rmse), 4),
    "mape": round(float(mape), 4),
    "r2": round(float(r2), 4),
    "train_samples": len(X_train),
    "test_samples": len(X_test)
}
(ARTIFACTS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

# 5. Save feature importance
feat_imp = [{"feature": f, "importance": float(imp)} for f, imp in zip(FEATURE_COLUMNS, model.feature_importances_)]
feat_imp.sort(key=lambda x: x["importance"], reverse=True)
with (ARTIFACTS_DIR / "feature_importance.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["feature", "importance"])
    writer.writeheader()
    writer.writerows(feat_imp)

# 6. Save hourly history
df_energy.to_csv(DATA_DIR / "energy_consumption_hourly.csv", index=False)

print("All artifacts successfully exported for production serving!")